# EduCasual Final Project Analysis

## Clear Question

**Does an increase in online engagement prior to an assessment cause an increase in student performance on that assessment?**

This notebook is the presentation layer. The data processing, variable definitions, model estimation, and plotting functions all live in `src/educasual/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from educasual.config import ProjectPaths, load_analysis_config
from educasual.pipeline import build_panel_dataset
from educasual.models.fixed_effects import collect_model_summaries
from educasual.models.robustness import run_model_suite
from educasual.reporting import make_display_summary, pretty_variable_name
from educasual.visualization.plots import (
    plot_click_distribution,
    plot_decile_relationship,
    plot_heterogeneity_relationship,
)

paths = ProjectPaths()
config = load_analysis_config()

## Clean Panel Dataset

If the processed panel already exists, we load it. Otherwise, we build it from the raw OULAD CSV files in `data/raw/oulad/`.

In [ ]:
panel_path = paths.processed_data / "student_assessment_panel.csv"
if panel_path.exists():
    panel = pd.read_csv(panel_path)
else:
    panel = build_panel_dataset(paths.raw_data, panel_path)

panel.head()

In [ ]:
panel.rename(columns={
    "log_clicks_28d": pretty_variable_name("log_clicks_28d"),
    "is_low_ses": pretty_variable_name("is_low_ses"),
    "is_repeat_student": pretty_variable_name("is_repeat_student"),
})[["student_course_id", "id_assessment", "score", pretty_variable_name("log_clicks_28d"), pretty_variable_name("is_low_ses"), pretty_variable_name("is_repeat_student")]].describe(include="all")

## Main Fixed-Effects Model And Extensions

In [ ]:
results = run_model_suite(panel, config)
summary = collect_model_summaries(results)
make_display_summary(summary).sort_values(["model", "variable"]).reset_index(drop=True)

## Figures

Recommended final presentation:

1. distribution of pre-assessment engagement
2. main relationship using `Pre-assessment engagement (log(1 + clicks in previous 28 days))`
3. subgroup comparison for lower-SES students
4. subgroup comparison for gender
5. model table including non-banked and coursework-only robustness

In [ ]:
plot_click_distribution(panel)
plot_decile_relationship(panel)
plot_heterogeneity_relationship(panel)
plot_heterogeneity_relationship(panel, subgroup='is_female', subgroup_labels=('Male', 'Female'))

## Suggested Write-Up Structure

- One clear causal question.
- One clean panel dataset.
- One main fixed-effects model.
- Two or three robustness and subgroup analyses.
- A few clean figures that match the proposal.